In [ ]:
#-------------- step 1 : read with pandas --------------
#import the pandas libarary
import pandas as pd
#read the csv file into a dataframe
df = pd.read_csv("banking_support.csv")
#show the first 5 rows
print(df.head())
#show the last 5 rows
print(df.tail())
#show all rows
print(df)

In [ ]:
#------------ step 2 : understand the data----------
#shape returns (number of rows, number of columns)
print("shape of the dataset:")
print(df.shape)
print()
print("we have", df.shape[0], "rows and", df.shape[1], "columns.")

#columns names
print("the columns are:")
print(df.columns.tolist())

#data types of each column
print("the data types are:")
print(df.dtypes)

#detailed info about the dataframe
print("detailed info about the dataframe:")
print(df.info())

#describe the dataset
print("summary statistics of the dataset:")
print(df.describe())

#check if there are any missing values
print("missing values in each column:")
print(df.isnull().sum())

#look at one sample question and response to understand the content
print("a sample question")
print(df['questions'][0])
print("a sample response")
print(df['response'][0])
print()
print("category:")
print(df['category'][0])
print("priority:")
print(df['priority'][0])

Step 3 : EDA PART 1 -- Category Distribution
EDA stands for Exploratory data analysis.Before building any model,we visualize the data to understand it.

In [ ]:
#import the visulization libary
import matplotlib.pyplot as plt
print("imported matplotlib.pyplot as plt")

In [ ]:
#count how many rows belong to each category
print("count of rows for each category:")
print(df['category'].value_counts())
print()
print("total unique categories:")
print(df['category'].nunique())

In [ ]:
#plot a bar chart of the category distribution
plt.figure(figsize=(14,5))
df['category'].value_counts().plot(kind='bar',color='skyblue',edgecolor='black')
plt.title('Category Distribution',fontsize=16)
plt.xlabel('Category(Intent)')
plt.ylabel('Number of tickets')
plt.xticks(rotation=45,ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# list all the unique categories clearly with numbers
# FIX: the original code looped over every row (df['category'].tolist()), which repeats
# each category once per ticket. We only want the *unique* category names, and the
# count in the message should match the data (this dataset has 18 tickets across a
# handful of categories, not a fixed 19).
unique_categories = sorted(df['category'].unique().tolist())
print(f"all {len(unique_categories)} categories the chatbot will handle (from {df.shape[0]} tickets):")
print()
for i, category in enumerate(unique_categories, 1):
    print(f"{i:<2}. {category}")

Step 4 : EDA PART : 2 - Priority distribution and Question Length

we now look at two more things:
 1.how urgent are this  queries (priority column)
 2.how long are the customer questions ?(length of question text)

In [ ]:
#priority distribution
print("priority distribution:")
print(df['priority'].value_counts())
print()
#which categories have the highest priority ("High")
#FIX: the priority column stores values like 'High'/'Medium'/'Low'/'Urgent' (capitalized),
#so comparing against the lowercase 'high' never matched anything. Also compare against
#the most urgent level actually present, 'Urgent', in addition to 'High'.
print("categories with 'High' priority tickets:")
print(df[df['priority'] == 'High']['category'].tolist())
print()
print("categories with 'Urgent' priority tickets:")
print(df[df['priority'] == 'Urgent']['category'].tolist())

In [ ]:
#plot priority as a pie chart
priority_counts = df['priority'].value_counts()
colors = ['lightcoral', 'lightskyblue', 'lightgreen', 'gold']
plt.figure(figsize=(8, 8))
plt.pie(priority_counts, labels=priority_counts.index, autopct='%1.1f%%', startangle=90, colors=colors[:len(priority_counts)])
plt.title('Priority Distribution', fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
#compute the character length of each question
#FIX: the dataframe's column is called 'questions' (plural) - the original code
#referenced 'question' (singular), which doesn't exist and raises a KeyError.
df['question_length'] = df['questions'].apply(len)
print("question length statistics:")
print(df['question_length'].describe().round(1))
print()
print("shortest question:")
print(df.loc[df['question_length'].idxmin(), 'questions'])
print()
print("longest question:")
print(df.loc[df['question_length'].idxmax(), 'questions'])

In [ ]:
#plot question length distribution as a histogram
plt.figure(figsize=(10, 6))
plt.hist(df['question_length'], bins=10, color='skyblue', edgecolor='black')
plt.title('Question Length Distribution', fontsize=16)
plt.xlabel('Length of Question')
plt.ylabel('Number of Questions')
plt.tight_layout()
plt.show()

In [ ]:
#compute average question length by priority level
print("average question length by priority level:")
print(df.groupby('priority')['question_length'].mean().round(1))

step 5 : Text Preprocessing
Why do we need to prepare text
Raw text has noice that does not help the ML model:
i>capital letters: Account and account should be treated as the same word
ii>Punctuation:questions marks and commas carry no intent meaning
iii>Stopwords:words like my,the,is,what,should appear in every sentence and carry no useful information
we will clean the question column in three steps:
i>Convert to lowercase
ii>Remove punctuation using regular expression
iii>Remove stopwords using NLTK

In [ ]:
#import NLTK - Natural Language Toolkit
#this is the standard python library for text processing
import nltk
import re
nltk.download('stopwords',quiet=True)
nltk.download('punkt',quiet=True)
from nltk.corpus import stopwords
#NOTE: this cell needs an internet connection the first time it runs so NLTK can
#download the 'stopwords' dataset. If you get a LookupError, make sure you're online
#and re-run this cell.

In [ ]:
#see what English stopwords look like
english_stopwords = set(stopwords.words('english'))
print("total english stopwords:", len(english_stopwords))
print()
print("sample stopwords:")
print(sorted(list(english_stopwords))[:20])  # print first 20 stopwords sorted

In [ ]:
#define the text cleaning function
#this function will take a text string as input and return a cleaned version of the text
def clean_text(text):
    #convert to lowercase
    text = text.lower()
    #remove punctuation
    text = re.sub(r'[^a-z\s]', '', text)
    #remove stopwords
    words = text.split()
    words = [w for w in words if w not in english_stopwords]
    #join the remaining words back into a sentance
    cleaned = ' '.join(words)
    return cleaned

In [ ]:
#test the function on one example before applying to the full dataset
raw_question = "What is the capital of France?"
cleaned_question = clean_text(raw_question)
print("Original:", raw_question)
print("Cleaned:", cleaned_question)

In [ ]:
#applying the cleaning function to the full dataset
df['cleaned_questions'] = df['questions'].apply(clean_text)
print("cleaning applied to all rows")
print()
print("before and after for first 5 rows:")
print()
for i in range(5):
    print("Original:", df['questions'].iloc[i])
    print("Cleaned:", df['cleaned_questions'].iloc[i])
    print()

In [ ]:
#view the update dataframe new cleaned_questions column
print("Updated DataFrame:")
print(df[['questions', 'cleaned_questions', 'category']].head(10))

step 6 : TF - IDF Vectorization : Converting text to numbers
The core problem: Machine Learning models work with numbers,not words
The solution : TF-IDE converts each question into a raw of numbers
How TF-IDE works : 
i>Every unique word in the dataset become one column
ii>Every question became one raw
iii>The numbers at each cell = How important that word is for the question
iv>common words get low scores.Rare specific words get high scores
TF = Term Frequency:how often this word appears in this questions IDF = Inverse Document Frequency : how rare this word is across all question TF-IDF score = TF multiplied by IDF

In [ ]:
#import TfidfVectorizer from scikit-learn
#scikit-learn is the main machine learning library in python
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
#create the TfidfVectorizer
#max_features = 500 means we want to keep only the top 500 most frequent words
vectorizer = TfidfVectorizer(max_features=500)
#fit_transform does two things
#fit = learn the vocabulary from the questions
#transform=convert each question into a row of numbers
X = vectorizer.fit_transform(df['cleaned_questions'])

print("TF-IDF vectorization complete")
print()
print("shape of x (feature matrix):", X.shape)
print()
print(f'{X.shape[0]} rows = one row per question')
print(f'{X.shape[1]} columns = one column per unique word in the dataset')

In [ ]:
#define y - the target labels that the model will learn to predict
# y is category column from the original dataframe
y = df['category']
print("target variable y defined")
print()
#FIX: the dataset actually has df.shape[0] questions, not a hardcoded 19 - print the
#real count so this stays correct even if rows are added/removed later.
print(f"y contains the category for each of the {len(y)} questions")
print(y.tolist())

In [ ]:
#see which words the vectorizer learned from our dataset
feature_names = vectorizer.get_feature_names_out()
print("total words in vocabulary:", len(feature_names))
print()
print("first 30 words in alphabetical order:")
print(sorted(feature_names[:30]))

In [ ]:
#convert the matrix to a readable dataframe to understand it
import numpy as np
x_df = pd.DataFrame(X.toarray(), columns=feature_names)

#show the small portion -rows are questions , columns are words , values are tf-idf scores
print("TF-IDF matrix - first 5 rows, first 10 columns:")
print("Rows = questions, Columns = words, Values = tf-idf scores")
print(x_df.iloc[:5, :10])


In [ ]:
#show the top scoring words for the first question
first_question_scores = x_df.iloc[0]
top_words = first_question_scores[first_question_scores > 0].sort_values(ascending=False)
print("original question:", df['questions'][0])
#FIX: the cleaned column is called 'cleaned_questions' (plural), matching the column
#created in step 5 - the original code referenced the singular 'cleaned_question',
#which doesn't exist.
print("cleaned question:", df['cleaned_questions'][0])
print()
print("the tf-idf scores for this question are:")
print(top_words.head(10))  # show top 10 words with highest tf-idf scores

In [ ]:
#test the vectorizer on a complete new unseen question
#this simulates what happens when the chatbot receives a user query
new_question = 'my debit card is not working in the atm'
new_question_cleaned = clean_text(new_question)
new_question_vectorized = vectorizer.transform([new_question_cleaned])
print("new question (raw):", new_question)
print("new question (cleaned):", new_question_cleaned)
print("vectorized shape:", new_question_vectorized.shape)
print()
print("the question is now a row of numbers")
print("next we feed this into the ml model to predict the category")

step 7 : Train-Test Split
the textbook and exam analogy:imagine a student who studies from textbook and is then tested with the same exact question from the textbook.Scoring 100% tells you nothing about wheather the student actually understood the subject-they just may have memorized the answer.A fair test uses question the student has never seen before

Machine learning works the same way.if we train a model and test it on the same data,we cannot tell  if it has truly learned the underlying patterns or has  simply memorized the training example.this is called overfitting:the model performs great on data it has already seen,but poorly on new,unseen data

To get an honest estimate of how well our model will perform on new customer questions,we split our dataset into two pairs:
i>Training test(80%): the textbook the model studied from
ii> Test se(20%): the unseen exam question used only to check performance

In [ ]:
#import the train-test split function from scikit-learn
from sklearn.model_selection import train_test_split
#Split x(features) and y(labels) into training and test sets
#test size = 0.2 means 20 percent of rows go into the test set
#random state = 42 makes the split reproducible-everyone gets the same split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("training set size:", X_train.shape)
print("test set size:", X_test.shape)
print()
print(f'{X_train.shape[0]} questions to train on, {X_test.shape[0]} questions to test on.')

In [ ]:
#let's see exactly which categories ended up in training vs testing
#this uses the original index position to look back at df['category']
train_categories = sorted(df.loc[y_train.index, 'category'].tolist())
test_categories = sorted(df.loc[y_test.index, 'category'].tolist())
print("categories in training set:", train_categories)
print("categories in test set:", test_categories)
print()
#FIX: the original code just asserted (incorrectly, as a hardcoded comment) that the
#test categories never appear in training. With this dataset/random_state that isn't
#actually true, so we check it directly instead of hardcoding a claim.
missing_from_training = sorted(set(test_categories) - set(train_categories))
if missing_from_training:
    print("Notice: these test categories never appear in the training set:", missing_from_training)
    print("With only 18 tickets total, some categories have very few examples, so a")
    print("random split can easily leave a category out of the training data entirely.")
    print("This is a small-dataset challenge worth keeping in mind for step 10.")
else:
    print("Every category in the test set also appears in the training set.")
    print("Even so, with only 18 tickets total some categories have just 2-3 training")
    print("examples, which is very little for the model to learn from - keep this in")
    print("mind when we evaluate the model in step 10.")

step 8 : Train naive Bayes Model
It is a simple ,fast ,and surprisingly effective algorithm for text classification.it is based on probability:it looks at every word in a question and calculated how likely eacg category is ,given those words.for example ,if the word "locked" mostly appeared in "account" category question during training,then the new question containing "locked" gets a higher probability score for the account category.

itnis called naive because it makes a simplifying assuymption:it treat every word as independent of every other word(it does not consider word order or grammer).this assumption is technically wrong,but in practice it works very well for short text like customer support questions.

What does "fitting" a model mean ? fitting a model means showing the model the training questions along with their correct categories,and letting it calculate the word-to-category probabilities.once fitted,the model can use those probabilities to predict the category of a brand new question it has never seen

In [ ]:
#import the naive bayes classifier for text data
from sklearn.naive_bayes import MultinomialNB
#create the model
#FIX: the model was created as `model` but then fit/used as `nb_model`, which was
#never defined - that raised a NameError. Using one consistent name throughout: nb_model.
nb_model = MultinomialNB()
#fit(train) the model on training data only
#the model learns which words are associated with which categories
nb_model.fit(X_train, y_train)
print("naive bayes model trained successfully")
print("the model learned word-to-category patterns from", X_train.shape[0], "training questions")
print()
print("categories the model knows about (learned from the training set):")
print(sorted(nb_model.classes_.tolist()))

step 9:Make Prediction
Now that the model is trained,lets use it predict categories.first we run it on the test set(the unseen exam question)and compare predictions to the actual answers.Then,for the live hands on moment,we type a brand new question that was never in our dataset at all and watch the model classify in real time

In [ ]:
#predict categories for every question in the test set
y_pred = nb_model.predict(X_test)
#put actual vs predicted side by side so we can compare easily
comparison = pd.DataFrame({'Actual category': y_test, 'Predicted category': y_pred})
comparison

In [ ]:
#student's hands-on moment
#try changing the sentence below to any banking question and re-run this cell
#watch the model predict categories for a sentence it has never seen before
live_question = "I forgot my online banking password, how can I reset it?"
#run it through the exact pipeline we built earlier
#clean the text -> convert to tf-idf numbers -> predict
live_question_clean = clean_text(live_question)
live_question_vectorized = vectorizer.transform([live_question_clean])
live_question_pred = nb_model.predict(live_question_vectorized)[0]
#predict_proba gives the model's confidence for every category it knows
live_probabilities = nb_model.predict_proba(live_question_vectorized)[0]
top_3 = sorted(zip(nb_model.classes_, live_probabilities), key=lambda pair: -pair[1])[:3]
print("live question:", live_question)
print("cleaned question:", live_question_clean)
print()
#FIX: the original code printed an undefined variable `live_predictions` - the
#actual prediction is stored in `live_question_pred`.
print("predicted category:", live_question_pred)
print()
print("top 3 predicted categories with confidence scores:")
for category, score in top_3:
    print(f'{category:20s}{score*100:.1f}%')


step10:Evaluate Model

Now we measure how well the model actually did on the unseen test questions. We'll look at three things:
i>Accuracy - the overall percentage of test questions the model classified correctly
ii>Classification report - precision, recall, and f1-score broken down by category, so we can see if the model is good at some categories and bad at others
iii>Confusion matrix - a table showing exactly which categories get mixed up with which

In [ ]:
#import evaluation metrics from scikit-learn
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

#accuracy: percentage of correct predictions on the test set
accuracy = accuracy_score(y_test, y_pred)
print("accuracy on the test set:")
print(f"{accuracy*100:.1f}%  ({int(accuracy*len(y_test))} out of {len(y_test)} test questions correct)")

In [ ]:
#classification report: precision, recall, and f1-score for each category
#precision: of the questions the model predicted as category X, how many actually were X
#recall: of the questions that actually were category X, how many did the model catch
#f1-score: a balance between precision and recall
#zero_division=0 keeps sklearn from warning about categories with no predictions/support
print("classification report:")
print(classification_report(y_test, y_pred, zero_division=0))

In [ ]:
#confusion matrix: rows = actual category, columns = predicted category
#the diagonal cells are correct predictions; anything off the diagonal is a mix-up
labels = sorted(y.unique())
cm = confusion_matrix(y_test, y_pred, labels=labels)
cm_df = pd.DataFrame(cm, index=labels, columns=labels)
print("confusion matrix (rows=actual, columns=predicted):")
print(cm_df)

plt.figure(figsize=(8, 6))
plt.imshow(cm, cmap='Blues')
plt.title('Confusion Matrix', fontsize=16)
plt.xlabel('Predicted category')
plt.ylabel('Actual category')
plt.xticks(range(len(labels)), labels, rotation=45, ha='right')
plt.yticks(range(len(labels)), labels)
for i in range(len(labels)):
    for j in range(len(labels)):
        plt.text(j, i, cm[i, j], ha='center', va='center',
                 color='white' if cm[i, j] > cm.max()/2 else 'black')
plt.colorbar(label='Number of questions')
plt.tight_layout()
plt.show()

### A quick note before we go further

With only 18 tickets spread across 5 categories, the test set has just 3-4 questions - so
each single misclassification swings the accuracy score by roughly 25%. That's why the
numbers above can look noisy or low: this is a real limitation of tiny datasets, not
necessarily a mistake in the approach.

So far we've built an **intent classifier** - it predicts a category like 'Card Services'
or 'Loans'. But a real chatbot needs to actually **reply** to the customer, and the
notebook never used the `response` column at all yet. That's what we finish next.

step 11 : Building the Actual Chatbot - Retrieval-Based Responses

Predicting a category isn't the same as knowing what to say back. Looking at the data,
a single category can map to several different replies - `Account Inquiry` alone covers
checking a balance, finding an ATM, and downloading a statement, each with its own
response. So category prediction by itself isn't precise enough to pick a reply.

Instead we use a **retrieval-based** approach, which is how many small-data support
chatbots work in practice:
i>Clean and vectorize the customer's question the same way we did for training (TF-IDF)
ii>Compare it against every training question using cosine similarity (a score from 0 to 1
for how similar two TF-IDF vectors are)
iii>Find the single most similar training question
iv>Reply with that question's stored `response`, and also show the Naive Bayes category
prediction alongside it for context

If nothing in our tiny dataset is similar enough to the customer's question, the chatbot
should say so honestly instead of guessing - so we add a confidence threshold that falls
back to a safe 'let me connect you to a human agent' message.

In [ ]:
#cosine similarity measures how similar two TF-IDF vectors are (1.0 = identical, 0 = unrelated)
from sklearn.metrics.pairwise import cosine_similarity

#below this similarity score we don't trust the match enough to answer automatically
CONFIDENCE_THRESHOLD = 0.25

def get_bot_response(user_question, threshold=CONFIDENCE_THRESHOLD):
    """
    Given a raw customer question, returns a dict with:
      - predicted_category: the Naive Bayes category prediction
      - category_confidence: the model's confidence in that category (0-1)
      - matched_question: the most similar question in the training data
      - similarity: how similar the matched question is (0-1)
      - response: the reply text the chatbot would send
    """
    cleaned = clean_text(user_question)
    vectorized = vectorizer.transform([cleaned])

    #category prediction (from step 8's trained model)
    predicted_category = nb_model.predict(vectorized)[0]
    category_confidence = nb_model.predict_proba(vectorized)[0].max()

    #find the most similar question we've seen before, and reuse its response
    similarities = cosine_similarity(vectorized, X)[0]
    best_idx = similarities.argmax()
    best_similarity = similarities[best_idx]

    if best_similarity < threshold:
        response = ("I'm not confident I understood that correctly. "
                     "Let me connect you with a human agent who can help.")
    else:
        response = df.iloc[best_idx]['response']

    return {
        'predicted_category': predicted_category,
        'category_confidence': round(float(category_confidence), 2),
        'matched_question': df.iloc[best_idx]['questions'],
        'similarity': round(float(best_similarity), 2),
        'response': response,
    }

print("get_bot_response() is ready")

In [ ]:
#try the chatbot on a handful of questions it has never seen before
sample_questions = [
    "my debit card is not working in the atm",
    "I forgot my online banking password, how can I reset it?",
    "can I check my balance please",
    "I need to send money to my friend",
    "what are your mortgage rates",
    "what's the weather like today",  # something totally unrelated, to test the fallback
]

for q in sample_questions:
    result = get_bot_response(q)
    print("Customer:", q)
    print(f"  -> predicted category: {result['predicted_category']} (confidence {result['category_confidence']*100:.0f}%)")
    print(f"  -> closest known question: \"{result['matched_question']}\" (similarity {result['similarity']*100:.0f}%)")
    print("  Bot:", result['response'])
    print()

step 12 : Save the Trained Chatbot

Training only has to happen once. We save the fitted vectorizer, the Naive Bayes model,
and the ticket data (needed for the retrieval step above) to disk with `joblib`, so any
other script can load them instantly without retraining.

In [ ]:
import joblib

joblib.dump(vectorizer, 'chatbot_vectorizer.pkl')
joblib.dump(nb_model, 'chatbot_nb_model.pkl')
df.to_csv('chatbot_reference_data.csv', index=False)

print("saved 3 files:")
print(" - chatbot_vectorizer.pkl   (fitted TF-IDF vectorizer)")
print(" - chatbot_nb_model.pkl     (trained Naive Bayes model)")
print(" - chatbot_reference_data.csv  (cleaned tickets used for response retrieval)")

step 13 : Chat With the Bot Interactively

This cell runs a live chat loop right in the notebook. Type a banking question and press
Enter to get a response; type `quit` to stop. This is the finished, end-to-end chatbot:
raw customer text in, a real reply out.

In [ ]:
#run this cell, then type a question and press Enter. Type 'quit' to exit the chat.
print("Banking Support Bot - type 'quit' to exit\n")
while True:
    user_input = input("You: ")
    if user_input.strip().lower() in ('quit', 'exit'):
        print("Bot: Thanks for chatting - goodbye!")
        break
    result = get_bot_response(user_input)
    print("Bot:", result['response'])
    print()

### Project summary

What this notebook now does, end to end:
i>Loads and explores 18 real-looking banking support tickets across 5 categories
ii>Cleans customer questions (lowercase, strip punctuation, remove stopwords)
iii>Converts questions to TF-IDF vectors
iv>Trains a Multinomial Naive Bayes classifier to predict ticket category
v>Evaluates that classifier honestly with accuracy, a classification report, and a confusion matrix
vi>Adds a retrieval layer that finds the closest known question by cosine similarity and
returns its real response - this is the part that makes it an actual chatbot rather than
just a category labeler
vii>Saves the trained artifacts so they can be reused without retraining
viii>Provides a live chat loop to talk to the finished bot

**Known limitation:** every part of this is trained on only 18 examples. Both the category
model and the retrieval matches will be shaky outside the phrasing patterns already in
the data (see the `atm`/weather examples in step 11). The architecture is production-shaped;
what it needs next is a much larger, more varied set of labeled tickets per category.